In [3]:
!git clone https://github.com/boragu/SRL-Code-Review-Study.git

%cd "/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin"

!mkdir -p data outputs


Cloning into 'SRL-Code-Review-Study'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 23 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 7.36 KiB | 7.36 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin


In [8]:
!pip -q install gdown

from pathlib import Path
import zipfile
import gdown

PROJECT_DIR = Path(
    "/content/SRL-Code-Review-Study/"
    "Store Sales - Time Series Forecasting/Hyunbin"
)
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = DATA_DIR / "store-sales-time-series-forecasting.zip"

gdown.download(
    id="1qUXEalsDiePFDUNuI4ThEihB0Jpr8VEt",
    output=str(ZIP_PATH),
    quiet=False,
)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(DATA_DIR)

print(list(DATA_DIR.rglob("*.csv")))

Downloading...
From: https://drive.google.com/uc?id=1qUXEalsDiePFDUNuI4ThEihB0Jpr8VEt
To: /content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/store-sales-time-series-forecasting.zip
100%|██████████| 22.4M/22.4M [00:00<00:00, 49.1MB/s]


[PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/stores.csv'), PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/oil.csv'), PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/test.csv'), PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/transactions.csv'), PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/holidays_events.csv'), PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/sample_submission.csv'), PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/train.csv')]


In [9]:
# =========================
# Store Sales Forecasting
# =========================

!pip -q install lightgbm

from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

SEED = 42
np.random.seed(SEED)

# 앞 셀에서 ZIP을 풀어 둔 폴더
PROJECT_DIR = Path(
    "/content/SRL-Code-Review-Study/"
    "Store Sales - Time Series Forecasting/Hyunbin"
)
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_file(filename):
    matches = list(DATA_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"{filename} 파일을 찾지 못했습니다.")
    return matches[0]

# -------------------------
# 1. 데이터 불러오기
# -------------------------
train = pd.read_csv(
    find_file("train.csv"),
    dtype={
        "id": "int32",
        "store_nbr": "int8",
        "family": "category",
        "sales": "float32",
        "onpromotion": "float32",
    },
)

test = pd.read_csv(
    find_file("test.csv"),
    dtype={
        "id": "int32",
        "store_nbr": "int8",
        "family": "category",
        "onpromotion": "float32",
    },
)

stores = pd.read_csv(find_file("stores.csv"))
oil = pd.read_csv(find_file("oil.csv"))
holidays = pd.read_csv(find_file("holidays_events.csv"))

for df in [train, test, oil, holidays]:
    df["date"] = pd.to_datetime(df["date"])

train["onpromotion"] = train["onpromotion"].fillna(0).astype("float32")
test["onpromotion"] = test["onpromotion"].fillna(0).astype("float32")

print("train:", train.shape)
print("test :", test.shape)

# -------------------------
# 2. 기본 특징 생성
# -------------------------
all_families = sorted(
    set(train["family"].astype(str)) | set(test["family"].astype(str))
)
family_to_code = {family: i for i, family in enumerate(all_families)}

for df in [train, test]:
    df["family_code"] = (
        df["family"].astype(str).map(family_to_code).astype("int16")
    )
    df["series_id"] = (
        df["store_nbr"].astype("int32") * 100
        + df["family_code"].astype("int32")
    ).astype("int32")

oil = oil.rename(columns={"dcoilwtico": "oil_price"})

all_dates = pd.date_range(
    min(train["date"].min(), test["date"].min()),
    max(train["date"].max(), test["date"].max()),
    freq="D",
)

oil = (
    oil.set_index("date")
    .reindex(all_dates)
    .rename_axis("date")
    .reset_index()
)

oil["oil_price"] = oil["oil_price"].ffill().bfill().astype("float32")

holidays["transferred"] = (
    holidays["transferred"].astype(str).str.lower().eq("true")
)
holidays = holidays[~holidays["transferred"]].copy()

national_holiday = (
    holidays[holidays["locale"] == "National"]
    .groupby("date")
    .size()
    .clip(upper=1)
)

local_holiday = (
    holidays[holidays["locale"] == "Local"]
    .groupby(["date", "locale_name"])
    .size()
    .clip(upper=1)
)

regional_holiday = (
    holidays[holidays["locale"] == "Regional"]
    .groupby(["date", "locale_name"])
    .size()
    .clip(upper=1)
)

work_day = (
    holidays[holidays["type"] == "Work Day"]
    .groupby("date")
    .size()
    .clip(upper=1)
)

type_to_code = {v: i for i, v in enumerate(sorted(stores["type"].unique()))}
city_to_code = {v: i for i, v in enumerate(sorted(stores["city"].unique()))}
state_to_code = {v: i for i, v in enumerate(sorted(stores["state"].unique()))}

def make_base_features(df):
    df = df.copy()
    df = df.merge(stores, on="store_nbr", how="left")
    df = df.merge(oil, on="date", how="left")

    df["oil_price"] = df["oil_price"].ffill().bfill().astype("float32")

    df["year"] = df["date"].dt.year.astype("int16")
    df["month"] = df["date"].dt.month.astype("int8")
    df["day"] = df["date"].dt.day.astype("int8")
    df["dayofweek"] = df["date"].dt.dayofweek.astype("int8")
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype("int8")
    df["is_weekend"] = (df["dayofweek"] >= 5).astype("int8")

    df["national_holiday"] = (
        df["date"].map(national_holiday).fillna(0).astype("int8")
    )

    local_index = pd.MultiIndex.from_frame(df[["date", "city"]])
    df["local_holiday"] = (
        local_index.map(local_holiday).fillna(0).astype("int8")
    )

    regional_index = pd.MultiIndex.from_frame(df[["date", "state"]])
    df["regional_holiday"] = (
        regional_index.map(regional_holiday).fillna(0).astype("int8")
    )

    df["work_day"] = df["date"].map(work_day).fillna(0).astype("int8")

    df["cluster"] = df["cluster"].fillna(-1).astype("int8")
    df["store_type_code"] = df["type"].map(type_to_code).astype("int8")
    df["city_code"] = df["city"].map(city_to_code).astype("int16")
    df["state_code"] = df["state"].map(state_to_code).astype("int16")

    return df

train_base = make_base_features(train)
test_base = make_base_features(test)

# -------------------------
# 3. Lag 특징 생성
# -------------------------
train_base = train_base.sort_values(["series_id", "date"]).reset_index(drop=True)
train_base["log_sales"] = np.log1p(train_base["sales"]).astype("float32")

for lag in [1, 7, 14, 28]:
    train_base[f"lag_{lag}"] = (
        train_base.groupby("series_id")["log_sales"]
        .shift(lag)
        .astype("float32")
    )

train_base["rolling_mean_7"] = (
    train_base.groupby("series_id")["log_sales"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
    .astype("float32")
)

train_base["rolling_mean_14"] = (
    train_base.groupby("series_id")["log_sales"]
    .transform(lambda x: x.shift(1).rolling(14).mean())
    .astype("float32")
)

feature_cols = [
    "store_nbr",
    "family_code",
    "cluster",
    "store_type_code",
    "city_code",
    "state_code",
    "onpromotion",
    "oil_price",
    "year",
    "month",
    "day",
    "dayofweek",
    "weekofyear",
    "is_weekend",
    "national_holiday",
    "local_holiday",
    "regional_holiday",
    "work_day",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
]

categorical_cols = [
    "store_nbr",
    "family_code",
    "cluster",
    "store_type_code",
    "city_code",
    "state_code",
    "month",
    "dayofweek",
]

model_df = train_base.dropna(
    subset=["lag_1", "lag_7", "lag_14", "lag_28", "rolling_mean_7", "rolling_mean_14"]
).copy()

# -------------------------
# 4. 검증 및 학습
# -------------------------
valid_dates = sorted(model_df["date"].unique())[-16:]

train_df = model_df[~model_df["date"].isin(valid_dates)]
valid_df = model_df[model_df["date"].isin(valid_dates)]

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 96,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "verbosity": -1,
    "seed": SEED,
    "num_threads": -1,
}

train_set = lgb.Dataset(
    train_df[feature_cols],
    label=train_df["log_sales"],
    categorical_feature=categorical_cols,
)

valid_set = lgb.Dataset(
    valid_df[feature_cols],
    label=valid_df["log_sales"],
    categorical_feature=categorical_cols,
)

model = lgb.train(
    params,
    train_set,
    num_boost_round=2500,
    valid_sets=[valid_set],
    valid_names=["validation"],
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(100),
    ],
)

best_iteration = model.best_iteration or 1000
print(f"Best iteration: {best_iteration}")

# -------------------------
# 5. 재귀 예측 함수
# -------------------------
def recursive_forecast(model, future_base, history_df):
    future_base = future_base.sort_values(["date", "series_id"]).copy()
    predictions = []

    last_logs = (
        history_df.sort_values(["series_id", "date"])
        .groupby("series_id")["log_sales"]
        .apply(lambda x: list(x.tail(28)))
        .to_dict()
    )

    for current_date in sorted(future_base["date"].unique()):
        batch = future_base[future_base["date"] == current_date].copy()

        batch["lag_1"] = [last_logs[sid][-1] for sid in batch["series_id"]]
        batch["lag_7"] = [last_logs[sid][-7] for sid in batch["series_id"]]
        batch["lag_14"] = [last_logs[sid][-14] for sid in batch["series_id"]]
        batch["lag_28"] = [last_logs[sid][-28] for sid in batch["series_id"]]

        batch["rolling_mean_7"] = [
            np.mean(last_logs[sid][-7:]) for sid in batch["series_id"]
        ]
        batch["rolling_mean_14"] = [
            np.mean(last_logs[sid][-14:]) for sid in batch["series_id"]
        ]

        pred_log = model.predict(batch[feature_cols], num_iteration=best_iteration)
        pred_log = np.maximum(pred_log, 0)

        batch["pred_log_sales"] = pred_log
        predictions.append(batch[["id", "series_id", "pred_log_sales"]])

        for sid, pred in zip(batch["series_id"], pred_log):
            last_logs[sid].append(float(pred))
            last_logs[sid] = last_logs[sid][-28:]

    return pd.concat(predictions, ignore_index=True)

# 검증 RMSLE
history_for_valid = train_base[train_base["date"] < min(valid_dates)]
valid_base = train_base[train_base["date"].isin(valid_dates)]

valid_pred = recursive_forecast(model, valid_base, history_for_valid)

valid_compare = valid_df[["id", "log_sales"]].merge(
    valid_pred[["id", "pred_log_sales"]],
    on="id",
    how="left",
)

valid_rmsle = np.sqrt(
    np.mean((valid_compare["log_sales"] - valid_compare["pred_log_sales"]) ** 2)
)

print(f"Validation RMSLE: {valid_rmsle:.5f}")

# -------------------------
# 6. 전체 데이터로 재학습 후 제출 파일 생성
# -------------------------
full_train_set = lgb.Dataset(
    model_df[feature_cols],
    label=model_df["log_sales"],
    categorical_feature=categorical_cols,
)

final_model = lgb.train(
    params,
    full_train_set,
    num_boost_round=best_iteration,
)

test_predictions = recursive_forecast(
    final_model,
    test_base,
    train_base,
)

submission = test_predictions[["id", "pred_log_sales"]].copy()
submission["sales"] = np.maximum(np.expm1(submission["pred_log_sales"]), 0)
submission = submission[["id", "sales"]].sort_values("id")

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("\nsubmission.csv 생성 완료")
print(submission.head())
print(f"\n저장 위치: {submission_path}")

train: (3000888, 6)
test : (28512, 5)
Training until validation scores don't improve for 100 rounds
[100]	validation's rmse: 0.39216
[200]	validation's rmse: 0.384243
[300]	validation's rmse: 0.380618
[400]	validation's rmse: 0.378484
[500]	validation's rmse: 0.377236
[600]	validation's rmse: 0.376208
[700]	validation's rmse: 0.375374
[800]	validation's rmse: 0.374901
[900]	validation's rmse: 0.374448
[1000]	validation's rmse: 0.374266
[1100]	validation's rmse: 0.373848
[1200]	validation's rmse: 0.373709
[1300]	validation's rmse: 0.373534
[1400]	validation's rmse: 0.373409
[1500]	validation's rmse: 0.373264
[1600]	validation's rmse: 0.373169
[1700]	validation's rmse: 0.373231
Early stopping, best iteration is:
[1693]	validation's rmse: 0.372976
Best iteration: 1693
Validation RMSLE: 0.40524

submission.csv 생성 완료
        id        sales
0  3000888     4.723586
1  3000889     0.014070
2  3000890     6.256052
3  3000891  2385.047033
4  3000892     0.004187

저장 위치: /content/SRL-Code-Review